# Lab 8: Prompting Techniques

## The map before the territory

Labs 9-15 each cover one way of shaping a prompt to get better results from an LLM:

- **Zero-shot** (lab 9) - just ask, no examples
- **Few-shot** (lab 10) - show a couple of worked examples before the real question, often to steer the output into a structured format
- **Chain-of-thought** (lab 11) - ask the model to reason step by step before answering
- **System instructions** (lab 12) - a persistent instruction that shapes every reply, independent of what the user asks

All of these are really about one thing: building the right **messages list** to send to the API (lab 15 goes deep on that). This lab builds the reusable piece underneath all of them - a template system for turning "a task plus some data" into a finished prompt string, without any live API call yet. The Reading tab explains string formatting and class state from scratch if you need a refresher before diving in.

## Exercise

**Goal:** build a small reusable *prompt template*: a piece of text with blanks such as `{review}` that you fill in with different values each time. Templates keep your prompts consistent. You write the method that fills the blanks and that complains clearly when one is forgotten.

## Words you will see

- **Prompt**: the text you send to a language model.
- **Template**: a prompt with named blanks in curly braces, e.g. `"Classify this review: {review}"`. The blank is called a *placeholder*.
- **Class / method / `self`**: a class bundles data and functions. A method is a function inside a class. `self` is the object itself, so `self.template` is the template text stored on it.
- **`**kwargs`**: lets a function accept any number of `name=value` arguments. Inside the function, `kwargs` is a dict, e.g. `render(review="Great!")` gives `kwargs == {"review": "Great!"}`.
- **Set**: a collection with no repeats and no order, e.g. `{"review"}`. You can subtract sets: `{"a", "b"} - {"a"}` is `{"b"}`.
- **`str.format`**: `"Hi {name}".format(name="Ana")` gives `"Hi Ana"`.
- **Regular expression**: a pattern for finding text. The constructor already uses one to find the `{names}` in the template. You do not need to write any regex.
- **`ValueError`**: the standard Python error for "you gave me a bad value". You make one with `raise ValueError("message")`.

### `PromptTemplate.render(self, **kwargs)`

`__init__` is already written. When you create `PromptTemplate("Classify the sentiment of this review: {review}")`, it saves the text in `self.template` and saves the set of placeholder names in `self.required_vars`. Here that is `{"review"}`. You only write `render`.

**Receives:** the variable values as keyword arguments, such as `render(review="Great product!")`. Extra arguments that the template does not use are allowed and are simply ignored by `format`.

**Must return:** a string, which is the template with every placeholder filled in. For the example above it returns `"Classify the sentiment of this review: Great product!"`.

**If something is missing:** it must not return anything. It must raise a `ValueError` whose message names the missing variable or variables. Calling `greeting.render()` with no arguments is an error because `review` is required.

**The idea in plain words:**

1. Work out which required names were not supplied. Subtract the names you were given (`kwargs.keys()`) from `self.required_vars`.
2. If anything is left over, raise a `ValueError` with a message that lists those names, for example `Missing required variable(s): ['review']`. Sorting the names (`sorted`) keeps the message tidy when several are missing.
3. Otherwise call `self.template.format(**kwargs)` and return the result. The `**` unpacks the dict back into `name=value` arguments.

**Edge cases the solution handles:** a template used twice, like `"{x} and {x}"`, needs `x` only once. Several missing variables are all reported in one error. A template with no placeholders needs no arguments.

**Check (by hand):** with `PromptTemplate("Hi {name}, you are {age}")`, `required_vars` is `{"name", "age"}`. `render(name="Ana", age=30)` gives `"Hi Ana, you are 30"`. `render(name="Ana")` raises a `ValueError` that mentions `age`.

### What the checks test

The cell below the class does three things:

- it checks that `required_vars` for the greeting template is `{"review"}` (this works already because `__init__` is written for you);
- it checks that `render(review="Great product!")` returns exactly `"Classify the sentiment of this review: Great product!"`;
- it calls `render()` with no arguments and checks that a `ValueError` is raised. If your code returns something or raises a different error, the check fails.

Seeing `All checks passed.` means all three held.

In [ ]:
import re


class PromptTemplate:
    def __init__(self, template: str):
        self.template = template
        self.required_vars = set(re.findall(r"\{(\w+)\}", template))

    def render(self, **kwargs) -> str:
        """Fill in the template's placeholders.

        Args:
            **kwargs: a value for each placeholder, e.g. review="Great product!".

        Returns:
            The finished prompt string, e.g.
            "Classify the sentiment of this review: Great product!".

        Raises:
            ValueError: if any name in self.required_vars is not in kwargs. The
            message should list the missing names.
        """
        # TODO:
        # 1. missing = the names in self.required_vars that are not in kwargs.keys()
        #    (set subtraction works here).
        # 2. If missing is not empty, raise ValueError with a message that
        #    includes the missing names (sorted() makes it tidy).
        # 3. Otherwise return self.template.format(**kwargs).
        raise NotImplementedError("PromptTemplate.render() is not implemented yet - replace the TODO with your code")


greeting = PromptTemplate("Classify the sentiment of this review: {review}")
assert greeting.required_vars == {"review"}, 'greeting.required_vars should be ' + repr({"review"}) + ", got " + repr(greeting.required_vars)
assert greeting.render(review="Great product!") == "Classify the sentiment of this review: Great product!", 'greeting.render(review="Great product!") should be ' + repr("Classify the sentiment of this review: Great product!") + ", got " + repr(greeting.render(review="Great product!"))

try:
    greeting.render()
    raise AssertionError("should have raised ValueError for a missing variable")
except ValueError:
    pass

print("All checks passed.")

## Solution

In [ ]:
class PromptTemplate:
    def __init__(self, template: str):
        self.template = template
        self.required_vars = set(re.findall(r"\{(\w+)\}", template))

    def render(self, **kwargs) -> str:
        missing = self.required_vars - kwargs.keys()
        if missing:
            raise ValueError(f"Missing required variable(s): {sorted(missing)}")
        return self.template.format(**kwargs)


greeting = PromptTemplate("Classify the sentiment of this review: {review}")
print(greeting.render(review="Great product!"))

## Real-world project (offline)

The `TEMPLATES` dict below builds two reusable `PromptTemplate`s - one for sentiment classification, one for summarizing - and renders each against real review sentences loaded from the dataset. Nothing here calls an LLM yet: it's just building and printing the exact prompt strings that labs 9-12 will actually send over the network.

In [ ]:
"""Load the shared real-world review dataset (embedded so this notebook needs no repo checkout)."""
import csv
import io

REVIEW_CSV = """sentence,label
the movie was absolutely wonderful and kept me hooked the whole time,1
i loved every minute of this film,1
the acting was superb and the story was touching,1
this was one of the best movies i have seen all year,1
the plot was engaging and the pacing was perfect,1
a brilliant film with outstanding performances,1
the soundtrack was beautiful and matched the mood perfectly,1
i would watch this movie again without hesitation,1
the director did an amazing job telling this story,1
this film exceeded all of my expectations,1
the characters felt real and the dialogue was sharp,1
a truly moving and unforgettable experience,1
the visuals were stunning and the effects were impressive,1
this is a masterpiece that everyone should see,1
the humor was clever and the timing was excellent,1
i was on the edge of my seat the entire film,1
the ending was satisfying and well earned,1
this movie made me laugh and cry at the same time,1
a wonderfully crafted story with great performances,1
the chemistry between the leads was fantastic,1
i really enjoyed the twist near the end,1
this film is a delightful surprise from start to finish,1
the cinematography was gorgeous throughout the movie,1
a heartwarming tale that left me smiling,1
the pacing kept the tension high without ever dragging,1
i cannot recommend this movie enough,1
the product works exactly as advertised and arrived quickly,1
i am very happy with this purchase,1
the quality is excellent for the price,1
this item exceeded my expectations in every way,1
the customer service team was friendly and helpful,1
setup was quick and the instructions were clear,1
this gadget has made my daily routine so much easier,1
the build quality feels sturdy and well made,1
i would definitely buy this product again,1
the battery life is impressive and lasts all day,1
shipping was fast and the packaging was excellent,1
this is by far the best purchase i made this year,1
the design is sleek and it works flawlessly,1
i appreciate how easy this product is to use,1
the price is fair given the quality you get,1
this laptop is fast reliable and great for work,1
the headphones sound amazing and are comfortable to wear,1
i love how lightweight and portable this device is,1
the app is intuitive and runs smoothly on my phone,1
this coffee maker brews a delicious cup every morning,1
the movie was boring and dragged on for way too long,0
i did not enjoy this film at all,0
the acting felt wooden and the story made no sense,0
this was one of the worst movies i have seen all year,0
the plot was confusing and the pacing was terrible,0
a disappointing film with weak performances,0
the soundtrack was forgettable and did not fit the mood,0
i would never watch this movie again,0
the director seemed to have no clear vision for this story,0
this film fell far short of my expectations,0
the characters felt flat and the dialogue was awkward,0
a frustrating and forgettable experience overall,0
the visuals were dull and the effects looked cheap,0
this movie is a mess that nobody should waste time on,0
the humor fell flat and the timing was off,0
i kept checking the clock throughout the entire film,0
the ending felt rushed and unearned,0
this movie made me want to leave the theater early,0
a poorly crafted story with unconvincing performances,0
the chemistry between the leads felt forced,0
i was annoyed by the predictable twist near the end,0
this film is a tedious slog from start to finish,0
the cinematography was flat and uninspired throughout,0
a dull tale that left me checking my phone,0
the pacing dragged and the tension never built,0
i cannot recommend this movie to anyone,0
the product broke after only two days of use,0
i am very unhappy with this purchase,0
the quality is poor for the price charged,0
this item fell short of my expectations in every way,0
the customer service team was rude and unhelpful,0
setup was confusing and the instructions were unclear,0
this gadget has made my daily routine more frustrating,0
the build quality feels flimsy and cheaply made,0
i would never buy this product again,0
the battery life is disappointing and drains quickly,0
shipping was slow and the packaging was damaged,0
this is by far the worst purchase i made this year,0
the design is clunky and it barely works,0
i am frustrated by how hard this product is to use,0
the price is too high given the poor quality,0
this laptop is slow unreliable and bad for work,0
the headphones sound tinny and are uncomfortable to wear,0
i hate how heavy and bulky this device is,0
the app is confusing and crashes on my phone,0
this coffee maker leaks and ruins my morning,0
the sequel was even better than the original film,1
i found the story dull and the characters uninteresting,0
the special effects were groundbreaking and beautifully done,1
the remote control stopped working within a week,0
this phone case fits perfectly and looks great,1
the instructions were missing several important steps,0
the keyboard feels responsive and is a joy to type on,1
the screen arrived scratched and the seller ignored my complaint,0
this backpack is durable and has plenty of storage,1
the movie relied on tired clichés and predictable jokes,0
the restaurant served delicious food with generous portions,1
our waiter was attentive and the service was outstanding,1
the pasta was cooked perfectly and full of flavor,1
this cafe has a cozy atmosphere and friendly staff,1
the steak was tender and cooked exactly how i asked,1
dessert was the highlight of an already fantastic meal,1
the wine pairing recommendations were spot on,1
i loved the fresh ingredients and creative menu,1
the bakery bread is soft and always baked fresh,1
this bistro has become my favorite spot in town,1
the book kept me turning pages late into the night,1
the author writes with wit and genuine emotional depth,1
i finished this novel in a single weekend it was that good,1
the world building in this book is rich and immersive,1
this memoir was honest funny and deeply moving,1
the mystery had twists i never saw coming,1
the poetry collection was beautiful and thought provoking,1
i recommend this book to anyone who loves a good story,1
the hotel room was spotless and beautifully decorated,1
the staff greeted us warmly and upgraded our room,1
the pool area was relaxing and well maintained,1
check in was smooth and the concierge was helpful,1
the breakfast buffet had a wonderful variety of fresh food,1
our stay at this resort exceeded every expectation,1
the view from the balcony was absolutely breathtaking,1
the gym and spa facilities were top notch,1
this hotel is perfect for a relaxing weekend getaway,1
the car handles smoothly and the ride is comfortable,1
fuel efficiency on this vehicle has been impressive,1
the dealership made the buying process quick and painless,1
this vacuum cleaner picks up dirt with ease,1
the blender is powerful and easy to clean,1
these shoes are comfortable and have held up well,1
the jacket is warm stylish and fits perfectly,1
the restaurant food was bland and arrived cold,0
our waiter seemed annoyed and service was painfully slow,0
the pasta was overcooked and lacked any real flavor,0
this cafe was noisy and the staff were dismissive,0
the steak was tough and clearly overdone,0
dessert was stale and ruined an otherwise average meal,0
the wine pairing recommendations were completely off,0
i disliked the stale ingredients and boring menu,0
the bakery bread is dry and often stale,0
this bistro has become a place i actively avoid,0
the book dragged and i never finished reading it,0
the author writes with clumsy prose and shallow characters,0
i abandoned this novel halfway through out of boredom,0
the world building in this book felt lazy and inconsistent,0
this memoir was dull repetitive and emotionally flat,0
the mystery was predictable from the very first page,0
the poetry collection felt uninspired and forgettable,0
i would not recommend this book to anyone,0
the hotel room was dirty and poorly maintained,0
the staff ignored us and refused to help with our request,0
the pool area was filthy and badly maintained,0
check in took forever and the concierge was unhelpful,0
the breakfast buffet had stale food and few options,0
our stay at this resort fell far below expectations,0
the view from the balcony was blocked by construction,0
the gym and spa facilities were outdated and dirty,0
this hotel is a poor choice for any kind of getaway,0
the car rattles constantly and the ride is uncomfortable,0
fuel efficiency on this vehicle has been disappointing,0
the dealership made the buying process stressful and confusing,0
this vacuum cleaner barely picks up any dirt,0
the blender is weak and difficult to clean,0
these shoes are uncomfortable and fell apart quickly,0
the jacket is thin flimsy and does not fit well,0
"""

def load_reviews():
    reader = csv.DictReader(io.StringIO(REVIEW_CSV))
    sentences, labels = [], []
    for row in reader:
        sentences.append(row["sentence"])
        labels.append(int(row["label"]))
    return sentences, labels

sentences, labels = load_reviews()
print(f"Loaded {len(sentences)} real review sentences.")


In [ ]:
TEMPLATES = {
    "classify_sentiment": PromptTemplate(
        "Classify the sentiment of this review as positive or negative. "
        "Reply with a single word.\n\nReview: {review}"
    ),
    "summarize": PromptTemplate(
        "Summarize this review in one short sentence.\n\nReview: {review}"
    ),
}

for name, template in TEMPLATES.items():
    print(f"--- {name} ---")
    for review in sentences[:2]:
        print(template.render(review=review))
        print()